# Run ALPR Vehicle System in Google Colab

Use this notebook to run the project on a Colab GPU instead of your laptop. Recommended runtime: **Runtime → Change runtime type → T4 GPU**.

You can either:
1. Upload the project folder to Google Drive, or
2. Upload a `.zip` of the project directly into Colab.

The notebook expects these files to exist inside the project folder:
- `app.py`
- `src/`
- `models/yolo_plate/best.pt`
- `yolov8n.pt`


In [ ]:
# 1) Check GPU
!nvidia-smi

import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


## Option A: Use Project from Google Drive

Upload your project folder to Drive, then edit `PROJECT_DIR` below if needed.

In [ ]:
# 2A) Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Change this if your folder is in a different Drive location.
PROJECT_DIR = '/content/drive/MyDrive/alpr-vehicle-system'
%cd $PROJECT_DIR
!ls -la


## Option B: Upload a Project ZIP Instead

Skip this section if you already used Drive above. If using this option, upload a zip named something like `alpr-vehicle-system.zip`.

In [ ]:
# 2B) Upload and unzip project folder
# Skip if you mounted Google Drive in the previous section.

# Jump to root folder and wipe out the old files
%cd /content
!rm -rf /content/alpr-vehicle-system*

from google.colab import files
uploaded = files.upload()

import os, zipfile
zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
assert zip_names, 'Please upload a .zip file of the project.'

zip_path = zip_names[0]
extract_dir = '/content/alpr-vehicle-system'
os.makedirs(extract_dir, exist_ok=True)
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

# If the zip contains a top-level folder, auto-detect it.
candidates = []
for root, dirs, files_in_root in os.walk(extract_dir):
    if 'app.py' in files_in_root and 'src' in dirs:
        candidates.append(root)
assert candidates, 'Could not find app.py and src/ inside the uploaded zip.'

PROJECT_DIR = candidates[0]
%cd $PROJECT_DIR
!ls -la


In [ ]:
# 3) Install dependencies for Colab
# Colab already includes torch in most GPU runtimes, so install the project libraries without forcing the local CUDA index.
!apt-get install -q tesseract-ocr
!pip install -q ultralytics easyocr gradio opencv-python-headless pandas numpy matplotlib scikit-learn pyyaml pytesseract


In [ ]:
# 4) Verify required files & download yolov8n.pt if missing
from pathlib import Path

required_paths = [
    Path('app.py'),
    Path('src/pipeline.py'),
    Path('src/ocr/plate_ocr.py'),
    Path('models/yolo_plate/best.pt'),
]

missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError('Missing required files:\n' + '\n'.join(missing))

# yolov8n.pt is auto-downloaded by ultralytics on first run
if not Path('yolov8n.pt').exists():
    from ultralytics import YOLO
    YOLO('yolov8n.pt')
    print('Downloaded yolov8n.pt')

print('All required files found.')


## Run Direct Video Processing

Use this path if you just want to process a video and download the output, without opening the Gradio UI.

In [ ]:
# 5) Upload a video to process
from google.colab import files
uploaded_video = files.upload()
video_files = [name for name in uploaded_video if name.lower().endswith(('.mp4', '.mov', '.avi', '.mkv'))]
assert video_files, 'Please upload a video file.'
VIDEO_PATH = video_files[0]
print('Video selected:', VIDEO_PATH)


In [ ]:
# 6) Process the video
# If you want more accuracy, try frame_skip=2. If you want more speed, try frame_skip=4 or 5.
from src.pipeline import process_video

out_path, detections_df = process_video(VIDEO_PATH, frame_skip=3)
print('Processed video:', out_path)
display(detections_df)


In [ ]:
# 7) Preview and download outputs
from IPython.display import HTML, display
from base64 import b64encode
from pathlib import Path
from google.colab import files

if Path(out_path).exists():
    mp4 = open(out_path, 'rb').read()
    data_url = 'data:video/mp4;base64,' + b64encode(mp4).decode()
    display(HTML(f'<video width="720" controls><source src="{data_url}" type="video/mp4"></video>'))
    files.download(out_path)

log_path = Path('outputs/logs/detections.csv')
if log_path.exists():
    files.download(str(log_path))


## Optional: Launch the Gradio UI in Colab

Run this if you prefer using the app interface. Colab will show a public Gradio link.

In [ ]:
# 8) Patch launch mode for Colab, then run the app
from pathlib import Path

app_path = Path('app.py')
app_text = app_path.read_text(encoding='utf-8')
app_text = app_text.replace('demo.launch()', 'demo.launch(share=True, debug=True)')
Path('app_colab.py').write_text(app_text, encoding='utf-8')

!python app_colab.py


## Notes

- Colab runtimes reset, so keep your trained `models/yolo_plate/best.pt` in Drive.
- If processing is still slow, increase `frame_skip` in cell 6.
- If plates are missed, lower `frame_skip` to `2` and rerun.
- If the Gradio cell keeps running, that is normal; stop it manually when done.